In [2]:
import pandas as pd
import numpy as np

clean = pd.read_csv("data/rentals_clean.csv")
print(clean.shape)   
clean.head()

(7538, 10)


,house_type,locality,city,area,beds,bathrooms,balconies,furnishing,area_rate,rent
0,"2 BHK Flat for Rent in Oberoi Woods, Goregaon ...",Goregaon East,Mumbai,897.0,2,2,0,Semi-Furnished,134.0,120000.0
1,"1 BHK Flat for Rent in Sapphire Lakeside, Powa...",Powai,Mumbai,490.0,1,1,0,Semi-Furnished,82.0,40000.0
2,1 BHK House for Rent in Mundhwa Pune,Mundhwa,Pune,550.0,1,1,0,Unfurnished,22.0,12000.0
3,"2 BHK Flat for Rent in Hingna, Nagpur",Hingna,Nagpur,1000.0,2,2,0,Unfurnished,8.0,8000.0
4,1 BHK Flat for Rent in Unique Star Harsh Vihar...,Mira Road,Mumbai,595.0,1,1,0,Unfurnished,25.0,15000.0


In [3]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(clean, test_size=0.2, random_state=42)

print("Train flats:", train_df.shape[0])
print("Test flats: ", test_df.shape[0])

Train flats: 6030
Test flats:  1508


In [4]:
from sklearn.metrics import mean_squared_error

# Learn the average rent per locality, using ONLY training data
locality_avg = train_df.groupby("locality")["rent"].mean()

# Predict each test flat by looking up its locality's average
baseline_pred = test_df["locality"].map(locality_avg)

# Some test localities might not appear in training - fill those with overall average
baseline_pred = baseline_pred.fillna(train_df["rent"].mean())

# Measure how wrong we are, in rupees
baseline_rmse = np.sqrt(mean_squared_error(test_df["rent"], baseline_pred))
print(f"Baseline RMSE: ₹{baseline_rmse:,.0f}")

Baseline RMSE: ₹41,434


In [5]:
# Features (inputs) the model learns from, and the target (what we predict)
features = ["area", "beds", "bathrooms", "balconies", "furnishing", "locality", "city"]
target = "rent"

# Turn text columns into numbers (one-hot encoding)
X_train = pd.get_dummies(train_df[features])
X_test  = pd.get_dummies(test_df[features])

# Train and test must have identical columns - force them to match
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = train_df[target]
y_test  = test_df[target]

print("Feature columns after encoding:", X_train.shape[1])

Feature columns after encoding: 1762


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# --- Model 1: Linear Regression 
lr = LinearRegression()
lr.fit(X_train, y_train)                      # learn the pattern
lr_pred = lr.predict(X_test)                   # guess rents for unseen flats
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
print(f"Linear Regression RMSE: ₹{lr_rmse:,.0f}")

# --- Model 2: Random Forest 
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
print(f"Random Forest RMSE: ₹{rf_rmse:,.0f}")

Linear Regression RMSE: ₹31,580
Random Forest RMSE: ₹27,762


In [7]:
print("--- Results (lower RMSE = better) ---")
print(f"Baseline (locality avg):  ₹{baseline_rmse:,.0f}")
print(f"Linear Regression:        ₹{lr_rmse:,.0f}")
print(f"Random Forest:            ₹{rf_rmse:,.0f}")

improvement = (baseline_rmse - rf_rmse) / baseline_rmse * 100
print(f"\nBest model beats baseline by {improvement:.0f}%")

--- Results (lower RMSE = better) ---
Baseline (locality avg):  ₹41,434
Linear Regression:        ₹31,580
Random Forest:            ₹27,762

Best model beats baseline by 33%


In [8]:
def verdict(actual_rent, predicted_rent, tolerance=0.10):
    if actual_rent > predicted_rent * (1 + tolerance):
        return "Overpriced"
    elif actual_rent < predicted_rent * (1 - tolerance):
        return "Good deal"
    return "Fairly priced"

# Try it on the first 10 hidden test flats
results = test_df.head(10).copy()
results["predicted"] = rf.predict(X_test.head(10)).round()
results["verdict"] = results.apply(
    lambda row: verdict(row["rent"], row["predicted"]), axis=1
)

print(results[["city", "locality", "beds", "rent", "predicted", "verdict"]].to_string())

           city              locality  beds      rent  predicted     verdict
6838     Nagpur        Narendra Nagar     2   30000.0    21140.0  Overpriced
1047  Bangalore  Attibele Anekal Road     3   25000.0    41172.0   Good deal
6801     Nagpur        Narendra Nagar     2    5500.0    13400.0   Good deal
5435       Pune          Sinhgad Road     3   21000.0    32170.0   Good deal
2114       Pune         Kalyani Nagar     3   60000.0    49820.0  Overpriced
2357  Bangalore               Kannuru     5  100000.0   138170.0   Good deal
3852       Pune              Lohegaon     1    6000.0     5222.0  Overpriced
2847     Mumbai            Prabhadevi     3  280000.0   135920.0  Overpriced
4690       Pune              Hadapsar     2   20000.0    17638.0  Overpriced
2333  New Delhi          Palam Colony     1    5000.0     8020.0   Good deal


In [10]:
import joblib
joblib.dump(rf, "models/rent_model.pkl")
joblib.dump(X_train.columns.tolist(), "models/model_columns.pkl")
print("Model saved!")

Model saved!
